In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print(f"Loading data for analysis...")
X_train = np.load(MACHINE_A_TRAIN_X)

# Create a clean DataFrame
df = pd.DataFrame(X_train, columns=MACHINE_A_FEATURES)
print(f"Data Shape: {df.shape}")

In [ ]:
# Standardize data first (VIF requires scaled data for accurate results)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
df_scaled = pd.DataFrame(X_scaled, columns=MACHINE_A_FEATURES)

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = df_scaled.columns
vif_data["VIF"] = [variance_inflation_factor(df_scaled.values, i) 
                   for i in range(len(df_scaled.columns))]

print("\n--- VIF Analysis (Redundancy Check) ---")
print(vif_data.sort_values(by="VIF", ascending=False))

# INTERPRETATION:
# If 'Mean Power' and 'Variance' both have huge VIFs (e.g., > 100),
# it proves mathematically that one is a perfect duplicate of the other.

In [ ]:
# Run PCA
pca = PCA()
pca.fit(X_scaled)

# Calculate explained variance
explained_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 4), explained_variance, marker='o', linestyle='--')
plt.axhline(y=0.95, color='r', linestyle=':', label='95% Explained Variance')
plt.title('PCA: How many features do we actually need?')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.xticks([1, 2, 3, 4])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n--- PCA Report ---")
for i, var in enumerate(explained_variance):
    print(f"Component {i+1}: Explains {var:.2%} of the data variance")